# Notebook 2: Reusable LongBench Data Preprocessing Pipeline

**Dataset:** zai-org/LongBench, 16 English LongBench v1 tasks

This notebook converts the raw LongBench records into one clean, reproducible, model-ready format for every KV-cache method.

It performs:

1. Raw data extraction and schema normalization
2. Cleaning and duplicate removal
3. Official LongBench prompt construction
4. Exact target-model tokenization
5. 32K-token middle truncation
6. Fixed 50-items-per-task development split
7. Stable JSONL export
8. Before-and-after Data Quality Index reporting

This notebook prepares evaluation inputs. It does not train a model, run inference, or compress a KV cache.


## Processing contract

Every method in the project must load the same processed records. Method owners should not recreate prompts, choose another truncation rule, or create another development split.

The authoritative model input is the processed prompt field.

Output files:

| File | Purpose |
| --- | --- |
| processed_longbench.jsonl | All cleaned records with a split field |
| processed_longbench_dev.jsonl | Fixed 800-record development subset |
| dev_ids.json | Stable IDs for the development subset |
| cleaning_log.csv | Rows retained and removed at each stage |
| dqi_before_after.csv | Quality dimensions before and after cleaning |
| manifest.json | Configuration, tokenizer, source, and schema metadata |


## 1. Setup and configuration

Change project settings only in the next cell.

The notebook does not apply a chat template. This preserves the official LongBench prompt format. If the team later chooses a chat template, it must be applied consistently to every method.


In [1]:
import hashlib
import json
import zipfile
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from IPython.display import display

HF_REPO = "zai-org/LongBench"
SEED = 42
DEV_PER_TASK = 50
CONTEXT_CAP_TOKENS = 32_000
MODEL_NAME = "meta-llama/Llama-3.1-8B-Instruct"

RAW_DIR = Path("longbench_raw")
OUT_DIR = Path("processed_longbench")
OUT_DIR.mkdir(parents=True, exist_ok=True)

PROMPT_URL = (
    "https://raw.githubusercontent.com/THUDM/LongBench/main/"
    "LongBench/config/dataset2prompt.json"
)

TASKS = OrderedDict({
    "narrativeqa": ("Single-Doc QA", "F1"),
    "qasper": ("Single-Doc QA", "F1"),
    "multifieldqa_en": ("Single-Doc QA", "F1"),
    "hotpotqa": ("Multi-Doc QA", "F1"),
    "2wikimqa": ("Multi-Doc QA", "F1"),
    "musique": ("Multi-Doc QA", "F1"),
    "gov_report": ("Summarization", "ROUGE-L"),
    "qmsum": ("Summarization", "ROUGE-L"),
    "multi_news": ("Summarization", "ROUGE-L"),
    "trec": ("Few-shot", "Classification score"),
    "triviaqa": ("Few-shot", "F1"),
    "samsum": ("Few-shot", "ROUGE-L"),
    "passage_count": ("Synthetic", "Accuracy"),
    "passage_retrieval_en": ("Synthetic", "Accuracy"),
    "lcc": ("Code", "Edit similarity"),
    "repobench-p": ("Code", "Edit similarity"),
})

QUALITY_WEIGHTS = {
    "completeness": 0.30,
    "validity": 0.25,
    "uniqueness": 0.15,
    "consistency": 0.15,
    "fitness": 0.15,
}

assert len(TASKS) == 16
assert sum(QUALITY_WEIGHTS.values()) == 1.0
print(f"Configured {len(TASKS)} tasks")


Configured 16 tasks


## 2. Load the raw LongBench records

The Hugging Face repository provides data.zip. The code downloads it once, caches it locally, and reads one JSONL file per task.

The task name is assigned from the file being loaded. This gives the pipeline a stable local task key even when the original metadata uses a different naming convention.


### ---- 2.1 Download and load raw files ----

In [2]:
def ensure_raw_files():
    if RAW_DIR.exists() and list(RAW_DIR.rglob("*.jsonl")):
        return

    from huggingface_hub import hf_hub_download

    zip_path = hf_hub_download(
        repo_id=HF_REPO,
        filename="data.zip",
        repo_type="dataset",
    )
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(RAW_DIR)


def load_task(task):
    matches = list(RAW_DIR.rglob(f"{task}.jsonl"))
    if not matches:
        raise FileNotFoundError(f"{task}.jsonl not found under {RAW_DIR}")

    rows = []
    with matches[0].open(encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if line.strip():
                row = json.loads(line)
                row["task"] = task
                row["source_file"] = str(matches[0].relative_to(RAW_DIR))
                row["source_line"] = line_number
                rows.append(row)
    return pd.DataFrame(rows)


ensure_raw_files()
raw = pd.concat([load_task(task) for task in TASKS], ignore_index=True)

print("Raw records:", len(raw))
display(raw.groupby("task").size().rename("n_raw").to_frame())


Raw records: 3750


,n_raw
task,
2wikimqa,200
gov_report,200
hotpotqa,200
lcc,500
multi_news,200
multifieldqa_en,150
musique,200
narrativeqa,200
passage_count,200


## 3. Load the official LongBench prompt templates

The prompt templates are loaded from the official LongBench repository rather than manually recreated.

Each template uses the context and input fields. The generated prompt is stored in the processed record so that every method receives the same model input.


### ---- 3.1 Download official prompt templates ----

In [3]:
response = requests.get(PROMPT_URL, timeout=60)
response.raise_for_status()
PROMPTS = response.json()

missing_prompts = sorted(set(TASKS) - set(PROMPTS))
if missing_prompts:
    raise KeyError(f"Missing official prompts for: {missing_prompts}")

print(f"Loaded {len(TASKS)} official prompts")


Loaded 16 official prompts


## 4. Normalize fields and clean records

Cleaning is conservative:

- Normalize text and line endings.
- Convert answers to a non-empty list of strings.
- Require a record ID, context, and answer list.
- Keep empty input where it is part of the official task design.
- Keep only the 16 English tasks used by this project.
- Remove duplicate IDs and duplicate task-document-question records, retaining the first occurrence.

The cleaning log records every removal so the process is auditable.


### ---- 4.1 Normalization helpers ----

In [4]:
def clean_text(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return ""
    text = str(value).replace("\x00", "")
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text.strip()


def normalize_answers(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return []
    values = list(value) if isinstance(value, (list, tuple, np.ndarray)) else [value]
    return [clean_text(item) for item in values if clean_text(item)]


def normalize_row(row):
    row = row.copy()
    row["input"] = clean_text(row.get("input", ""))
    row["context"] = clean_text(row.get("context", ""))
    row["answers"] = normalize_answers(row.get("answers"))
    row["_id"] = clean_text(row.get("_id", ""))
    row["language"] = clean_text(row.get("language", "")).lower()
    row["dataset"] = clean_text(row.get("dataset", ""))
    return row


normalized = pd.DataFrame(
    [normalize_row(row) for _, row in raw.iterrows()]
)
print("Rows after normalization:", len(normalized))

# Keep this snapshot for the before-cleaning DQI.
raw_quality = normalized.copy()


Rows after normalization: 3750


### ---- 4.2 Apply cleaning and save the audit log ----

In [5]:
working = normalized.copy()
cleaning_rows = []


def log_stage(stage, before, after, reason):
    cleaning_rows.append({
        "stage": stage,
        "rows_before": int(before),
        "rows_after": int(after),
        "rows_removed": int(before - after),
        "reason": reason,
    })


before = len(working)
working = working[working["_id"].ne("")].copy()
log_stage("required_record_id", before, len(working), "record ID must be present")

before = len(working)
working = working[working["context"].ne("")].copy()
log_stage("required_context", before, len(working), "context must be present")

before = len(working)
working = working[working["answers"].map(len).gt(0)].copy()
log_stage("required_answers", before, len(working), "at least one answer is required")

before = len(working)
working = working[working["language"].eq("en")].copy()
log_stage("english_only", before, len(working), "project scope is English tasks")

before = len(working)
working = working[working["task"].isin(TASKS)].copy()
log_stage("known_task", before, len(working), "task must be in the project registry")

before = len(working)
working = working.drop_duplicates(subset=["_id"], keep="first").copy()
log_stage("duplicate_id", before, len(working), "retain first record per ID")

working["context_hash"] = working["context"].map(
    lambda text: hashlib.sha256(text.encode("utf-8")).hexdigest()
)

before = len(working)
working = working.drop_duplicates(
    subset=["task", "context_hash", "input"],
    keep="first",
).copy()
log_stage(
    "duplicate_task_context_input",
    before,
    len(working),
    "retain first duplicate document-question pair",
)

working = working.reset_index(drop=True)
cleaning_log = pd.DataFrame(cleaning_rows)
cleaning_log.to_csv(OUT_DIR / "cleaning_log.csv", index=False)

print("Clean records:", len(working))
display(cleaning_log)


Clean records: 2731


,stage,rows_before,rows_after,rows_removed,reason
0,required_record_id,3750,3750,0,record ID must be present
1,required_context,3750,3750,0,context must be present
2,required_answers,3750,3750,0,at least one answer is required
3,english_only,3750,2750,1000,project scope is English tasks
4,known_task,2750,2750,0,task must be in the project registry
5,duplicate_id,2750,2750,0,retain first record per ID
6,duplicate_task_context_input,2750,2731,19,retain first duplicate document-question pair


## 5. Build official prompts

The prompt is constructed after cleaning and before truncation.

The prompt field is the common model input used by the full-KV baseline, eviction methods, quantization methods, KV-CAR, TurboQuant, and ShadowKV.


### ---- 5.1 Format prompts using official templates ----

In [6]:
def format_official_prompt(row):
    template = PROMPTS[row["task"]]
    return template.format(
        context=row["context"],
        input=row["input"],
    )


working["prompt_untruncated"] = working.apply(
    format_official_prompt,
    axis=1,
)
working["context_words"] = working["context"].str.split().str.len()
working["input_words"] = working["input"].str.split().str.len()
working["prompt_words"] = working["prompt_untruncated"].str.split().str.len()

display(
    working[
        ["task", "context_words", "prompt_words", "prompt_untruncated"]
    ].head(3)
)


,task,context_words,prompt_words,prompt_untruncated
0,narrativeqa,22686,22753,"You are given a story, which can be either a n..."
1,narrativeqa,23306,23377,"You are given a story, which can be either a n..."
2,narrativeqa,5391,5458,"You are given a story, which can be either a n..."


## 6. Load the exact target tokenizer

The EDA used an approximate tokenizer when the Llama tokenizer was unavailable. This preprocessing notebook uses the tokenizer named in the project configuration.

If access to the gated tokenizer is unavailable, fix the environment rather than silently substituting another tokenizer for final KV-cache accounting.


### ---- 6.1 Load the exact tokenizer ----

In [7]:
from huggingface_hub import login

login()

In [8]:
from huggingface_hub import get_token
from transformers import AutoTokenizer

hf_token = get_token()

if hf_token is None:
    raise RuntimeError("No Hugging Face token found.")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True,
    token=hf_token,
)

print("Loaded tokenizer:", MODEL_NAME)
print("Tokenizer class:", tokenizer.__class__.__name__)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Loaded tokenizer: meta-llama/Llama-3.1-8B-Instruct
Tokenizer class: PreTrainedTokenizerFast


### ---- 6.2 Apply token-level middle truncation ----

In [9]:
def middle_truncate_ids(token_ids, max_tokens):
    if len(token_ids) <= max_tokens:
        return token_ids, False

    left_count = max_tokens // 2
    right_count = max_tokens - left_count
    return token_ids[:left_count] + token_ids[-right_count:], True


def process_prompt(prompt):
    token_ids = tokenizer.encode(prompt, add_special_tokens=False)
    kept_ids, was_truncated = middle_truncate_ids(
        token_ids,
        CONTEXT_CAP_TOKENS,
    )
    processed_prompt = tokenizer.decode(
        kept_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )
    return {
        "prompt": processed_prompt,
        "prompt_tokens_before": len(token_ids),
        "prompt_tokens_after": len(kept_ids),
        "truncated": bool(was_truncated),
        "truncation_method": (
            "middle_keep_head_tail" if was_truncated else "none"
        ),
    }


processed_prompt_df = pd.DataFrame(
    working["prompt_untruncated"].map(process_prompt).tolist(),
    index=working.index,
)
working = pd.concat([working, processed_prompt_df], axis=1)

print("Rows requiring truncation:", int(working["truncated"].sum()))
print("Maximum processed tokens:", int(working["prompt_tokens_after"].max()))


Rows requiring truncation: 82
Maximum processed tokens: 32000


### ---- 6.3 Summarize truncation by task ----

In [10]:
 truncation_summary = (
    working.groupby("task")
    .agg(
        n_records=("_id", "size"),
        mean_tokens_before=("prompt_tokens_before", "mean"),
        p95_tokens_before=(
            "prompt_tokens_before",
            lambda s: s.quantile(0.95),
        ),
        max_tokens_before=("prompt_tokens_before", "max"),
        n_truncated=("truncated", "sum"),
    )
    .reindex(TASKS.keys())
)
truncation_summary["pct_truncated"] = (
    100
    * truncation_summary["n_truncated"]
    / truncation_summary["n_records"]
)
display(truncation_summary.round(2))
truncation_summary.to_csv(
    OUT_DIR / "preprocessing_truncation_summary.csv"
)


,n_records,mean_tokens_before,p95_tokens_before,max_tokens_before,n_truncated,pct_truncated
task,,,,,,
narrativeqa,182.0,30198.40,64172.85,65369.0,79.0,43.41
qasper,200.0,5087.46,9226.10,21275.0,0.0,0.00
multifieldqa_en,149.0,6944.62,13237.20,14997.0,0.0,0.00
hotpotqa,200.0,12853.24,16368.00,16395.0,0.0,0.00
2wikimqa,200.0,7167.24,14535.25,16385.0,0.0,0.00
musique,200.0,15615.51,16385.05,16404.0,0.0,0.00
gov_report,200.0,10274.34,20580.60,51426.0,3.0,1.50
qmsum,200.0,13915.58,26142.10,30437.0,0.0,0.00
multi_news,200.0,2641.05,6145.95,13968.0,0.0,0.00


## 7. Create the fixed 800-record development subset

The development subset contains exactly 50 records per task, selected with a fixed seed.

The exported dev_ids.json file is the source of truth. Every method must use this same ID list for development comparisons.


### ---- 7.1 Deterministic 50-per-task development subset ----

In [12]:
print("Expected tasks:", len(TASKS))
print("Tasks found in working:", working["task"].nunique())
print("Development parts created:", len(dev_parts))
print("Development records created:", len(dev))

print("\nRecords per task in working:")
display(
    working["task"]
    .value_counts()
    .sort_index()
    .rename("n_records")
    .to_frame()
)

print("\nTasks missing from working:")
print(sorted(set(TASKS.keys()) - set(working["task"].unique())))

print("\nUnexpected tasks:")
print(sorted(set(working["task"].unique()) - set(TASKS.keys())))

Expected tasks: 16
Tasks found in working: 14
Development parts created: 14
Development records created: 700

Records per task in working:


,n_records
2wikimqa,200
gov_report,200
hotpotqa,200
multi_news,200
multifieldqa_en,149
musique,200
narrativeqa,182
passage_count,200
passage_retrieval_en,200
qasper,200



Tasks missing from working:
['lcc', 'repobench-p']

Unexpected tasks:
[]


In [13]:
# ---- 4.2 Apply cleaning rules and save the cleaning log ----

working = normalized.copy()
cleaning_rows = []


def log_stage(stage, before, after, reason):
    cleaning_rows.append(
        {
            "stage": stage,
            "rows_before": int(before),
            "rows_after": int(after),
            "rows_removed": int(before - after),
            "reason": reason,
        }
    )


# 1. Remove records without a valid record ID
before = len(working)

working = working[
    working["_id"].ne("")
].copy()

log_stage(
    "required_record_id",
    before,
    len(working),
    "record ID must be present",
)


# 2. Remove records without context
before = len(working)

working = working[
    working["context"].ne("")
].copy()

log_stage(
    "required_context",
    before,
    len(working),
    "context must be present",
)


# 3. Remove records without reference answers
before = len(working)

working = working[
    working["answers"].map(len).gt(0)
].copy()

log_stage(
    "required_answers",
    before,
    len(working),
    "at least one answer is required",
)


# 4. Keep English tasks and the selected code-completion tasks.
# lcc and repobench-p are part of the official 16-task project scope,
# but their language field may not be recorded as "en".

CODE_TASKS = {
    "lcc",
    "repobench-p",
}

before = len(working)

working = working[
    working["language"].eq("en")
    | working["task"].isin(CODE_TASKS)
].copy()

log_stage(
    "language_scope",
    before,
    len(working),
    "retain English tasks and selected code-completion tasks",
)


# 5. Keep only tasks included in the official project registry
before = len(working)

working = working[
    working["task"].isin(TASKS)
].copy()

log_stage(
    "known_task",
    before,
    len(working),
    "task must be included in the project registry",
)


# 6. Remove duplicate record IDs
before = len(working)

working = working.drop_duplicates(
    subset=["_id"],
    keep="first",
).copy()

log_stage(
    "duplicate_id",
    before,
    len(working),
    "retain the first record for each record ID",
)


# 7. Create a document hash for duplicate-document detection
working["context_hash"] = working["context"].map(
    lambda text: hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()
)


# 8. Remove duplicate task-document-question combinations
before = len(working)

working = working.drop_duplicates(
    subset=[
        "task",
        "context_hash",
        "input",
    ],
    keep="first",
).copy()

log_stage(
    "duplicate_task_context_input",
    before,
    len(working),
    "retain the first duplicate document-question pair",
)


# 9. Reset the dataframe index
working = working.reset_index(drop=True)


# 10. Save the cleaning log
cleaning_log = pd.DataFrame(cleaning_rows)

cleaning_log.to_csv(
    OUT_DIR / "cleaning_log.csv",
    index=False,
)


# 11. Display the results
print("Clean records:", len(working))

display(cleaning_log)

print("\nRecords per task after cleaning:")

display(
    working["task"]
    .value_counts()
    .sort_index()
    .rename("n_records")
    .to_frame()
)

Clean records: 3731


,stage,rows_before,rows_after,rows_removed,reason
0,required_record_id,3750,3750,0,record ID must be present
1,required_context,3750,3750,0,context must be present
2,required_answers,3750,3750,0,at least one answer is required
3,language_scope,3750,3750,0,retain English tasks and selected code-complet...
4,known_task,3750,3750,0,task must be included in the project registry
5,duplicate_id,3750,3750,0,retain the first record for each record ID
6,duplicate_task_context_input,3750,3731,19,retain the first duplicate document-question pair



Records per task after cleaning:


,n_records
2wikimqa,200
gov_report,200
hotpotqa,200
lcc,500
multi_news,200
multifieldqa_en,149
musique,200
narrativeqa,182
passage_count,200
passage_retrieval_en,200


## 8. Compute the before-and-after Data Quality Index

The DQI is a project-defined diagnostic score from 0 to 100. It is not an official LongBench metric.

The consistency check verifies English language, present metadata, and known task membership. It does not require the original dataset string to equal the local task key because code-task metadata can use different naming conventions.

Fitness uses exact prompt token counts and describes operational suitability for the 32K experiment.


### ---- 8.1 Prepare raw normalized data for the before-cleaning DQI ----

In [16]:
# ---- Build prompts and apply exact-tokenizer middle truncation ----

# Build the official LongBench prompt
def format_official_prompt(row):
    template = PROMPTS[row["task"]]

    return template.format(
        context=row["context"],
        input=row["input"],
    )


working["prompt_untruncated"] = working.apply(
    format_official_prompt,
    axis=1,
)


# Calculate word-count statistics
working["context_words"] = (
    working["context"]
    .str.split()
    .str.len()
)

working["input_words"] = (
    working["input"]
    .str.split()
    .str.len()
)

working["prompt_words"] = (
    working["prompt_untruncated"]
    .str.split()
    .str.len()
)


# Token-level middle truncation
def middle_truncate_ids(token_ids, max_tokens):
    if len(token_ids) <= max_tokens:
        return token_ids, False

    left_count = max_tokens // 2
    right_count = max_tokens - left_count

    return (
        token_ids[:left_count]
        + token_ids[-right_count:],
        True,
    )


def process_prompt(prompt):
    token_ids = tokenizer.encode(
        prompt,
        add_special_tokens=False,
    )

    kept_ids, was_truncated = middle_truncate_ids(
        token_ids,
        CONTEXT_CAP_TOKENS,
    )

    processed_prompt = tokenizer.decode(
        kept_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )

    return {
        "prompt": processed_prompt,
        "prompt_tokens_before": len(token_ids),
        "prompt_tokens_after": len(kept_ids),
        "truncated": bool(was_truncated),
        "truncation_method": (
            "middle_keep_head_tail"
            if was_truncated
            else "none"
        ),
    }


# Process every cleaned record
processed_prompt_df = pd.DataFrame(
    working["prompt_untruncated"]
    .map(process_prompt)
    .tolist(),
    index=working.index,
)


# Add processed prompt information to working
working = pd.concat(
    [
        working,
        processed_prompt_df,
    ],
    axis=1,
)


print(
    "Rows requiring truncation:",
    int(working["truncated"].sum()),
)

print(
    "Maximum processed tokens:",
    int(working["prompt_tokens_after"].max()),
)

print(
    "Column created:",
    "prompt_tokens_after" in working.columns,
)

Rows requiring truncation: 88
Maximum processed tokens: 32000
Column created: True


In [17]:
# ---- 8. Data Quality Index: Before and After Cleaning ----

# Confirm that exact token processing has already been completed
if "prompt_tokens_after" not in working.columns:
    raise RuntimeError(
        "Run the prompt-formatting, tokenizer, and middle-truncation cells first."
    )


# Prepare raw data for the before-cleaning DQI
raw_quality = raw_quality.copy()

raw_quality["context_hash"] = raw_quality["context"].map(
    lambda text: hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()
)

raw_quality["prompt_untruncated"] = raw_quality.apply(
    format_official_prompt,
    axis=1,
)

raw_quality["prompt_tokens_before"] = (
    raw_quality["prompt_untruncated"]
    .map(
        lambda prompt: len(
            tokenizer.encode(
                prompt,
                add_special_tokens=False,
            )
        )
    )
)


def nonempty_text(value):
    return isinstance(value, str) and bool(value.strip())


def quality_flags(df, token_col):
    result = pd.DataFrame(index=df.index)

    # 1. Completeness
    result["completeness"] = (
        df["_id"].map(nonempty_text)
        & df["context"].map(nonempty_text)
        & df["answers"].map(
            lambda values: (
                isinstance(values, list)
                and len(values) > 0
            )
        )
    )

    # 2. Validity
    result["validity"] = (
        df["context"].map(nonempty_text)
        & df["answers"].map(
            lambda values: (
                isinstance(values, list)
                and all(
                    nonempty_text(answer)
                    for answer in values
                )
            )
        )
        & ~df["context"].str.contains(
            chr(0xFFFD),
            regex=False,
            na=False,
        )
        & df["length"].map(
            lambda value: (
                pd.notna(value)
                and float(value) >= 0
            )
        )
    )

    # 3. Uniqueness
    duplicate_ids = df["_id"].duplicated(
        keep="first"
    )

    duplicate_pairs = df.duplicated(
        subset=[
            "task",
            "context_hash",
            "input",
        ],
        keep="first",
    )

    result["uniqueness"] = ~(
        duplicate_ids
        | duplicate_pairs
    )

    # 4. Consistency
    # English tasks must have language="en".
    # Code tasks are retained even if their language
    # metadata is not recorded as "en".
    CODE_TASKS = {
        "lcc",
        "repobench-p",
    }

    language_is_valid = (
        df["language"]
        .astype(str)
        .str.lower()
        .eq("en")
        | df["task"].isin(CODE_TASKS)
    )

    result["consistency"] = (
        language_is_valid
        & df["dataset"].map(nonempty_text)
        & df["task"].isin(TASKS)
    )

    # 5. Fitness
    # Checks whether the processed prompt is within
    # the agreed 32K-token limit.
    result["fitness"] = (
        df[token_col]
        .le(CONTEXT_CAP_TOKENS)
    )

    return result


def dqi_score(flags):
    dimensions = list(QUALITY_WEIGHTS)

    scores = flags[dimensions].mean()

    score = 100 * sum(
        QUALITY_WEIGHTS[name] * scores[name]
        for name in dimensions
    )

    return scores, float(score)


# Calculate quality flags
before_flags = quality_flags(
    raw_quality,
    "prompt_tokens_before",
)

after_flags = quality_flags(
    working,
    "prompt_tokens_after",
)


# Calculate DQI scores
before_dimensions, before_dqi = dqi_score(
    before_flags
)

after_dimensions, after_dqi = dqi_score(
    after_flags
)


# Create before-and-after DQI report
dqi_report = pd.DataFrame(
    {
        "before_cleaning": before_dimensions,
        "after_cleaning": after_dimensions,
    }
)

dqi_report.loc[
    "DQI",
    "before_cleaning",
] = before_dqi / 100

dqi_report.loc[
    "DQI",
    "after_cleaning",
] = after_dqi / 100


# Display DQI results as percentages
display(
    (dqi_report * 100).round(2)
)

print(
    f"Before-cleaning DQI: {before_dqi:.2f}/100"
)

print(
    f"After-cleaning DQI: {after_dqi:.2f}/100"
)


# Save the DQI report
dqi_report.to_csv(
    OUT_DIR / "dqi_before_after.csv"
)

,before_cleaning,after_cleaning
completeness,100.00,100.00
validity,99.55,99.54
uniqueness,99.49,100.00
consistency,100.00,100.00
fitness,97.49,100.00
DQI,99.43,99.89


Before-cleaning DQI: 99.43/100
After-cleaning DQI: 99.89/100


## 9. Build the stable JSONL schema

The schema keeps original fields required for evaluation plus standardized processing metadata.

Downstream owners should not change the shared prompt, split, tokenizer, or truncation fields.


### ---- 9.1 Build stable records and export JSONL ----

In [19]:
# ---- Create development split and export processed JSONL files ----

# 1. Create exactly 50 development records per task
dev_parts = []

for task, group in working.groupby("task", sort=False):

    if len(group) < DEV_PER_TASK:
        raise ValueError(
            f"{task} has only {len(group)} records"
        )

    dev_parts.append(
        group.sample(
            n=DEV_PER_TASK,
            random_state=SEED,
        )
    )


dev = (
    pd.concat(dev_parts)
    .sort_values(
        ["task", "_id"]
    )
    .reset_index(drop=True)
)

dev_ids = dev["_id"].tolist()


# 2. Validate the development subset
assert len(dev) == DEV_PER_TASK * len(TASKS)

assert (
    dev["task"]
    .value_counts()
    .eq(DEV_PER_TASK)
    .all()
)


# 3. Save the development sample IDs
with (
    OUT_DIR / "dev_ids.json"
).open("w", encoding="utf-8") as handle:

    json.dump(
        {
            "seed": SEED,
            "samples_per_task": DEV_PER_TASK,
            "total_samples": len(dev_ids),
            "record_ids": dev_ids,
        },
        handle,
        indent=2,
    )


# 4. Add the split column to the working dataframe
working["split"] = np.where(
    working["_id"].isin(set(dev_ids)),
    "dev",
    "full",
)


print("Development records:", len(dev))

display(
    dev.groupby("task")
    .size()
    .rename("n_dev")
    .to_frame()
)


# 5. Create the stable output records
def stable_record(row):
    category, metric = TASKS[row["task"]]

    return {
        "record_id": row["_id"],
        "task": row["task"],
        "category": category,
        "metric": metric,
        "split": row["split"],
        "language": row["language"],
        "dataset": row["dataset"],
        "input": row["input"],
        "context": row["context"],
        "answers": row["answers"],
        "all_classes": row["all_classes"],
        "prompt": row["prompt"],
        "source_length_words": int(row["length"]),
        "context_words": int(row["context_words"]),
        "input_words": int(row["input_words"]),
        "prompt_words": int(row["prompt_words"]),
        "prompt_tokens_before": int(
            row["prompt_tokens_before"]
        ),
        "prompt_tokens_after": int(
            row["prompt_tokens_after"]
        ),
        "truncated": bool(row["truncated"]),
        "truncation_method": row[
            "truncation_method"
        ],
        "pipeline_version": (
            "longbench_preprocessing_v1"
        ),
    }


records = [
    stable_record(row)
    for _, row in working.iterrows()
]


# 6. Write JSONL files
def write_jsonl(path, rows):
    with Path(path).open(
        "w",
        encoding="utf-8",
    ) as handle:

        for row in rows:
            handle.write(
                json.dumps(
                    row,
                    ensure_ascii=False,
                )
                + "\n"
            )


write_jsonl(
    OUT_DIR / "processed_longbench.jsonl",
    records,
)

write_jsonl(
    OUT_DIR / "processed_longbench_dev.jsonl",
    [
        row
        for row in records
        if row["split"] == "dev"
    ],
)


# 7. Confirm the exported files
print(
    "Wrote full records:",
    len(records),
)

print(
    "Wrote dev records:",
    sum(
        row["split"] == "dev"
        for row in records
    ),
)

print(
    "Saved files in:",
    OUT_DIR.resolve(),
)

Development records: 800


,n_dev
task,
2wikimqa,50
gov_report,50
hotpotqa,50
lcc,50
multi_news,50
multifieldqa_en,50
musique,50
narrativeqa,50
passage_count,50


Wrote full records: 3731
Wrote dev records: 800
Saved files in: C:\Users\samru\Study\Study_Abroad\Universities\SJSU\Semester3\Data_298A_Project\Script\processed_longbench


### ---- 9.2 Validate the shared output schema ----

In [20]:
 required_output_fields = [
    "record_id", "task", "category", "metric", "split",
    "language", "dataset", "input", "context", "answers",
    "prompt", "source_length_words", "prompt_tokens_before",
    "prompt_tokens_after", "truncated", "truncation_method",
    "pipeline_version",
]

for record in records:
    missing = [
        field for field in required_output_fields
        if field not in record
    ]
    if missing:
        raise AssertionError(f"Missing fields: {missing}")
    if record["prompt_tokens_after"] > CONTEXT_CAP_TOKENS:
        raise AssertionError("Prompt exceeds token cap")
    if record["task"] not in TASKS:
        raise AssertionError("Unknown task")

assert len(records) == len({
    record["record_id"] for record in records
})
assert sum(record["split"] == "dev" for record in records) == (
    DEV_PER_TASK * len(TASKS)
)

print("Shared schema validation passed")


Shared schema validation passed


### ---- 9.3 Save a reproducibility manifest ----

In [21]:
manifest = {
    "pipeline_version": "longbench_preprocessing_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_dataset": HF_REPO,
    "official_prompt_source": PROMPT_URL,
    "model_name": MODEL_NAME,
    "tokenizer_name": MODEL_NAME,
    "context_cap_tokens": CONTEXT_CAP_TOKENS,
    "truncation_method": "middle_keep_head_tail",
    "seed": SEED,
    "dev_per_task": DEV_PER_TASK,
    "n_tasks": len(TASKS),
    "n_raw_records": int(len(raw)),
    "n_clean_records": int(len(working)),
    "n_dev_records": int(len(dev)),
    "before_dqi": round(before_dqi, 4),
    "after_dqi": round(after_dqi, 4),
    "output_schema": required_output_fields,
    "task_registry": {
        task: {"category": category, "metric": metric}
        for task, (category, metric) in TASKS.items()
    },
}

with (OUT_DIR / "manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(manifest, handle, indent=2)

print("Manifest saved to:", OUT_DIR / "manifest.json")


Manifest saved to: processed_longbench\manifest.json


## 10. Final handoff to method owners

Every method should use:

- processed_longbench_dev.jsonl for development runs
- processed_longbench.jsonl for full evaluation
- dev_ids.json to verify the fixed split
- manifest.json to verify tokenizer, cap, seed, and schema

Method owners may add runtime outputs such as generated answers, KV-cache statistics, latency, and memory usage. They should not change the shared prompt, split, or truncation fields.

### Completion checklist

- [ ] Official prompt templates loaded
- [ ] Exact target tokenizer loaded
- [ ] No duplicate record IDs remain
- [ ] Exactly 50 dev records exist per task
- [ ] Every processed prompt is at or below 32,000 tokens
- [ ] Full and dev JSONL files are present
- [ ] Cleaning log, DQI report, and manifest are present
- [ ] All method owners use the same processed files
